# Ropedia Academy — B3 · Camera pose: PnP & bundle adjustment

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChaoYue0307/ropedia-academy/blob/main/notebooks/B3.ipynb)

> **A working bundle-adjustment loop: jointly refine cameras + 3D points by minimizing total reprojection error, watching it converge from a noisy initialization.**
>
> 一个可运行的光束法平差循环：联合精修相机 + 3D 点以最小化总重投影误差，从含噪初值观察其收敛。

This is the lesson's core example — **self-contained and runnable end to end**. It builds toy tensors, performs the lesson's key computation, and prints a real result, so you learn the concept by executing it.

Colab's default runtime already includes `torch`, `numpy`, and `networkx`, so just press **Run all** — every cell should go green. Sizes are shrunk to run on CPU; swap in a real batch and the same code scales up.

🔗 Full lesson (with the interactive demo & key terms): https://chaoyue0307.github.io/ropedia-academy/lesson/B3

In [ ]:
import torch

# Reprojection error is geometry's universal currency. PnP = pose from 3D<->2D;
# Bundle Adjustment = jointly optimize ALL poses AND points to minimize it.
torch.manual_seed(0)
C, P = 3, 30                                       # cameras, 3D points
pts_true = torch.randn(P, 3) + torch.tensor([0,0,5.])
rot = torch.eye(3).expand(C, 3, 3)                 # (rotations fixed here for brevity)
trans_true = torch.randn(C, 3) * 0.1
def project(pts, R, tr):
    p = pts @ R.T + tr; return p[:, :2] / p[:, 2:3]
obs = torch.stack([project(pts_true, rot[c], trans_true[c]) for c in range(C)])

# Start from noisy guesses; refine by minimizing total reprojection error.
pts   = (pts_true + 0.3*torch.randn_like(pts_true)).requires_grad_()
trans = (trans_true + 0.1*torch.randn_like(trans_true)).requires_grad_()
opt = torch.optim.Adam([pts, trans], lr=0.02)
for _ in range(400):
    opt.zero_grad()
    pred = torch.stack([project(pts, rot[c], trans[c]) for c in range(C)])
    loss = ((pred - obs) ** 2).mean()              # sum_ij || x_ij - pi(K,R,t,X) ||^2
    loss.backward(); opt.step()
print("final reprojection error:", round(loss.item(), 6))
print("recovered 3D-point error:", round((pts - pts_true).norm().item(), 3), "(BA converged)")

### Where to go next

- Swap the toy tensors for a real batch and watch the shapes flow through.
- Open the matching lesson for the math and an interactive figure: https://chaoyue0307.github.io/ropedia-academy/lesson/B3
- Browse every notebook: https://github.com/ChaoYue0307/ropedia-academy/tree/main/notebooks